# Problem 1 - RNN, LSTM and GRU

Run the setup and validation cells first. The five full experiment cells are intentionally separate so each can be monitored and resumed.

In [ ]:
from pathlib import Path
import sys
import os
import subprocess
import numpy as np
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "problem1" else Path.cwd()
PROBLEM1 = ROOT / "problem1"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(PROBLEM1))
os.chdir(ROOT)

from problem1 import (
    run_smoke, run_har, run_adding, run_gradient_study,
    run_hidden_size_sweep, run_clipping_ablation, run_forget_bias_ablation,
)
from src.data import make_adding, validate_adding
from src.models import recurrent_parameter_explanation

print("Repository:", ROOT)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert ROOT.name == "2024SAMPLE_DNN_Assignment"


In [ ]:
x, y = make_adding(32, 50, 0)
validate_adding(x, y)
assert np.all(x[:, :, 1].sum(axis=1) == 2)
print(recurrent_parameter_explanation())
print(run_smoke())
assert recurrent_parameter_explanation()["equation_counts"] == {"RNN": 4736, "LSTM": 18944, "GRU": 14208}


## Full HAR comparison
Runs three seeds for RNN, LSTM, and GRU and writes rows to `results.csv`.

In [ ]:
import time
started = time.perf_counter()
har_results = run_har(False)
print("HAR runtime seconds:", time.perf_counter() - started)
assert len(har_results) == 9


## Full adding-problem comparison
Runs all models at T=50, 100, 200, and 400 with three seeds.

In [ ]:
started = time.perf_counter()
adding_results = run_adding(False)
print("Adding runtime seconds:", time.perf_counter() - started)
assert len(adding_results) == 36


## Gradient study
Runs the T=100 before/after gradient-norm experiment.

In [ ]:
started = time.perf_counter()
gradient_results = run_gradient_study(False)
print("Gradient runtime seconds:", time.perf_counter() - started)
assert set(gradient_results) == {"RNN", "LSTM", "GRU"}


## Hidden-size sweep
Runs LSTM and GRU at hidden sizes 16, 32, 64, and 128.

In [ ]:
started = time.perf_counter()
hidden_results = run_hidden_size_sweep(False)
print("Hidden-size runtime seconds:", time.perf_counter() - started)
assert len(hidden_results) == 24


## Individual ablation
Runs vanilla-RNN gradient clipping at T=200, norm 1.0 versus none. The
separate forget-bias cell below is available if course staff confirms the
original ID-based ablation rule still applies.

In [ ]:
started = time.perf_counter()
clipping_results = run_clipping_ablation(False)
print("Clipping runtime seconds:", time.perf_counter() - started)
assert len(clipping_results) == 6


In [ ]:
# Optional original-rubric ablation; enable only if course staff confirms it.
if False:
    forget_bias_results = run_forget_bias_ablation(False)
    assert len(forget_bias_results) == 6
